
# A2A Quickstart (Tavily + NVIDIA)
This Colab builds a **three-agent A2A system** with no Google Cloud dependencies:
- **Trending Topics Agent** (finds current trends using Tavily)
- **Trend Analyzer Agent** (quantifies a chosen trend)
- **Host Agent** (orchestrates the other two via the A2A protocol)
It also ships an **Inspector UI** to view agent cards and tasks.

**Requirements**: Tavily API key and NVIDIA API key only.<br>
**Note**: Do not forget to checkout the a2a inspector in the gradio ui app in the last cell.

In [ ]:
%pip install --upgrade --quiet pip wheel setuptools
%pip install --quiet --force-reinstall --no-cache-dir langgraph==0.4.6, tavily-python==0.5.0, langchain-nvidia-ai-endpoints==0.3.4


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 47.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
google-adk 1.16.0 requires tenacity<9.0.0,>=8.0.0, but you have tenacity 9.1.2 which is incompatible.
gradio 5.49.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.3 which is incompatible.


In [ ]:

#@title Install dependencies (quiet)
%%capture
!pip -q install fastapi==0.115.0 uvicorn==0.30.6 httpx==0.28.1   gradio==5.3.0 pydantic==2.11.1 tavily-python==0.5.0   langchain-nvidia-ai-endpoints==0.3.4 nest-asyncio==1.6.0


In [ ]:

#@title Enter your API keys
import os
TAVILY_API_KEY = ""  #@param {type:"string"}
NVIDIA_API_KEY = ""  #@param {type:"string"}

assert TAVILY_API_KEY, "Please paste your Tavily API key above"
assert NVIDIA_API_KEY, "Please paste your NVIDIA API key above"

os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY
os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY

# Optional model settings (you can tweak)
os.environ["NVIDIA_MODEL"] = "meta/llama-3.3-70b-instruct"
os.environ["TEMPERATURE"] = "0.2"
os.environ["TOP_P"] = "0.7"
os.environ["MAX_TOKENS"] = "1024"

print("Keys set. Ready.")


Keys set. Ready.


In [ ]:

#@title Shared helpers (LLM + Tavily)
import os, threading, uvicorn, asyncio, time, uuid, json, nest_asyncio
from typing import List, Dict, Any
from tavily import TavilyClient
from langchain_nvidia_ai_endpoints import ChatNVIDIA

def make_llm():
    return ChatNVIDIA(
        model=os.getenv("NVIDIA_MODEL", "meta/llama-3.3-70b-instruct"),
        api_key=os.getenv("NVIDIA_API_KEY", ""),
        temperature=float(os.getenv("TEMPERATURE", "0.2")),
        top_p=float(os.getenv("TOP_P", "0.7")),
        max_tokens=int(os.getenv("MAX_TOKENS", "1024")),
    )

_tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY", ""))

def tsearch(q: str, max_results: int = 5) -> Dict[str, Any]:
    return _tavily.search(q, max_results=max_results)

def llm_complete(prompt: str) -> str:
    llm = make_llm()
    out = []
    for c in llm.stream([{"role":"user","content": prompt}]):
        out.append(c.content or "")
    return "".join(out)

# quick smoke test
print("tavily found?", 'TavilyClient' in str(type(_tavily)))
print("nvidia llm ready? ok")


tavily found? True
nvidia llm ready? ok


In [ ]:

#@title A2A primitives (Task store + Agent Card helpers)
from fastapi import FastAPI, Request, BackgroundTasks
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field
from typing import Optional

class Task(BaseModel):
    id: str
    context_id: str
    state: str = "running"  # running | completed | failed
    created_at: float = Field(default_factory=lambda: time.time())
    updated_at: float = Field(default_factory=lambda: time.time())
    history: list = Field(default_factory=list)  # free-form debug logs
    artifacts: list = Field(default_factory=list)  # [{parts:[{type:'text', text:'...'}], sources:[...] }]

class InMemoryTaskStore:
    def __init__(self):
        self._tasks: Dict[str, Task] = {}
        self._lock = asyncio.Lock()

    async def create(self, context_id: Optional[str] = None) -> Task:
        async with self._lock:
            t = Task(id=uuid.uuid4().hex, context_id=context_id or uuid.uuid4().hex)
            self._tasks[t.id] = t
            return t

    async def update(self, task_id: str, **updates) -> Task:
        async with self._lock:
            t = self._tasks[task_id]
            for k,v in updates.items():
                setattr(t, k, v)
            t.updated_at = time.time()
            return t

    async def get(self, task_id: str) -> Task:
        return self._tasks[task_id]

    async def list(self) -> list[Task]:
        return list(self._tasks.values())

def make_agent_card(
    name: str, url: str, description: str, skills: list[dict], version: str = "1.0.0",
    transports: list[str] = ["http-json"]
):
    # A2A AgentCard essentials: capabilities, defaultInputModes, defaultOutputModes, skills[*].tags
    return {
        "protocolVersion": "1.0",
        "name": name,
        "description": description,
        "version": version,
        "url": url,
        "capabilities": {"streaming": False},
        "defaultInputModes": ["text/plain"],
        "defaultOutputModes": ["text/plain"],
        "preferredTransport": transports[0],
        "supportedTransports": transports,
        "skills": skills,
    }


In [ ]:

#@title Build worker agents (Trending + Analyzer)

def build_trending_app(port: int):
    app = FastAPI()
    store = InMemoryTaskStore()

    skills = [{
        "id": "trending.find",
        "name": "Find Trending Topics",
        "description": "Searches the web for trending topics in the last 24-48 hours and returns 3 items in JSON.",
        "tags": ["trends", "news", "search"],
        "examples": ["What's trending today?", "Show me current social trends"]
    }]
    card = make_agent_card(
        name="Trending Topics Agent",
        url=f"http://localhost:{port}",
        description="Finds current trending topics using Tavily web search.",
        skills=skills
    )

    @app.get("/.well-known/agent-card.json")
    async def well_known():
        return card

    @app.get("/a2a/card")
    async def compat_card():
        return card

    @app.get("/task/list")
    async def list_tasks():
        return [t.model_dump() for t in await store.list()]

    @app.get("/task/get")
    async def get_task(task_id: str):
        return (await store.get(task_id)).model_dump()

    async def _run_task(task_id: str, text: str):
        t = await store.get(task_id)
        try:
            t.history.append({"ts": time.time(), "event": "search:start", "query": "trending topics today site:news"})
            res = tsearch("trending topics today", max_results=8)
            urls = [r.get("url") for r in res.get("results", []) if r.get("url")]
            t.history.append({"ts": time.time(), "event": "search:done", "n": len(urls)})

            prompt = f"""You are a trends summarizer.
From the following urls and snippets, extract the top 3 distinct trending topics TODAY,
and return ONLY a JSON with this schema:
{{"trends":[{{"topic": "...","description":"...","reason":"..."}},{{...}},{{...}}]}}

Data:
{json.dumps(res)[:8000]}
"""
            t.history.append({"ts": time.time(), "event": "llm:start"})
            txt = llm_complete(prompt)
            t.history.append({"ts": time.time(), "event": "llm:done", "chars": len(txt)})
            await store.update(task_id, state="completed", artifacts=[{
                "parts":[{"type":"text","text": txt}],
                "sources": urls[:10]
            }])
        except Exception as e:
            await store.update(task_id, state="failed", artifacts=[{
                "parts":[{"type":"text","text": f"Error: {e}"}]
            }])

    @app.post("/message/send")
    async def message_send(req: Request, background_tasks: BackgroundTasks):
        body = await req.json()
        # Accept both A2A HTTP-JSON shape and a simple {"query": "..."} fallback
        msg = body.get("message") or {}
        parts = msg.get("parts") or []
        text = body.get("query") or (parts[0].get("text") if parts and parts[0].get("type")=="text" else "")
        if not text:
            text = "What's trending today?"

        t = await store.create(context_id=msg.get("contextId", uuid.uuid4().hex))
        background_tasks.add_task(_run_task, t.id, text)
        return {"task_id": t.id, "state": "running"}

    return app

def build_analyzer_app(port: int):
    app = FastAPI()
    store = InMemoryTaskStore()

    skills = [{
        "id": "trend.analyze",
        "name": "Analyze Trend",
        "description": "Given a trend/topic, collect quantitative metrics and produce a short, number-rich analysis.",
        "tags": ["analysis","metrics","statistics"],
        "examples": ["Analyze the '#AI' trend"]
    }]
    card = make_agent_card(
        name="Trend Analyzer Agent",
        url=f"http://localhost:{port}",
        description="Quantifies a given trend with concrete numbers and sources.",
        skills=skills
    )

    @app.get("/.well-known/agent-card.json")
    async def well_known():
        return card

    @app.get("/a2a/card")
    async def compat_card():
        return card

    @app.get("/task/list")
    async def list_tasks():
        return [t.model_dump() for t in await store.list()]

    @app.get("/task/get")
    async def get_task(task_id: str):
        return (await store.get(task_id)).model_dump()

    async def _run_task(task_id: str, trend: str):
        t = await store.get(task_id)
        try:
            q = f"statistics metrics numbers engagement growth geography for {trend}"
            t.history.append({"ts": time.time(), "event": "search:start", "query": q})
            res = tsearch(q, max_results=8)
            urls = [r.get("url") for r in res.get("results", []) if r.get("url")]
            t.history.append({"ts": time.time(), "event": "search:done", "n": len(urls)})
            prompt = f"""You are a quantitative analyst.
Using the web results below, write a concise analysis with **numbers** about the trend: {trend}.
- Include engagement metrics (views, shares), growth rates/timeline, geo splits if available.
- Cite key numbers inline in markdown bullets.
Keep it under 180 words.

Results:
{json.dumps(res)[:8000]}
"""
            t.history.append({"ts": time.time(), "event": "llm:start"})
            txt = llm_complete(prompt)
            t.history.append({"ts": time.time(), "event": "llm:done", "chars": len(txt)})
            await store.update(task_id, state="completed", artifacts=[{
                "parts":[{"type":"text","text": txt}],
                "sources": urls[:10]
            }])
        except Exception as e:
            await store.update(task_id, state="failed", artifacts=[{
                "parts":[{"type":"text","text": f"Error: {e}"}]
            }])

    @app.post("/message/send")
    async def message_send(req: Request, background_tasks: BackgroundTasks):
        body = await req.json()
        msg = body.get("message") or {}
        parts = msg.get("parts") or []
        trend = body.get("query") or (parts[0].get("text") if parts and parts[0].get("type")=="text" else "")
        if not trend:
            trend = "AI in social media"

        t = await store.create(context_id=msg.get("contextId", uuid.uuid4().hex))
        background_tasks.add_task(_run_task, t.id, trend)
        return {"task_id": t.id, "state": "running"}

    return app


In [ ]:
# @title Host agent (orchestrator) + A2A Inspector logs
from typing import Tuple
import asyncio, time, uuid, json
import httpx
from fastapi import FastAPI, BackgroundTasks, Request

ROUTER_LOGS = []  # list of dict logs only for the Inspector

def build_host_app(port: int, trending_url: str, analyzer_url: str):
    app = FastAPI()
    store = InMemoryTaskStore()  # assumes you defined this earlier

    skills = [{
        "id": "trends.report",
        "name": "Comprehensive Trend Report",
        "description": "Finds fresh trends, picks one, and returns a quantified analysis with sources.",
        "tags": ["orchestration", "routing", "report"],
        "examples": ["Analyze what's trending now and give me the numbers."]
    }]
    card = make_agent_card(  # assumes you defined this earlier
        name="Trend Analysis Host",
        url=f"http://localhost:{port}",
        description="Orchestrates the Trending and Analyzer agents via A2A task creation.",
        skills=skills
    )

    @app.get("/.well-known/agent-card.json")
    async def well_known():
        return card

    @app.get("/a2a/card")
    async def compat_card():
        return card

    @app.get("/task/list")
    async def list_tasks():
        return [t.model_dump() for t in await store.list()]

    @app.get("/task/get")
    async def get_task(task_id: str):
        return (await store.get(task_id)).model_dump()

    async def call_agent(base: str, text: str, max_wait: float = 120.0):
        async with httpx.AsyncClient(timeout=60.0) as client:
            payload = {
                "message": {
                    "role": "user",
                    "messageId": uuid.uuid4().hex,
                    "contextId": uuid.uuid4().hex,
                    "parts": [{"type": "text", "text": text}],
                }
            }
            ROUTER_LOGS.append({"ts": time.time(), "type": "send", "to": base, "text": text})
            r = await client.post(f"{base}/message/send", json=payload)
            j = r.json()
            task_id = j.get("task_id")
            assert task_id, f"Agent at {base} did not return task_id: {j}"
            t0 = time.time()
            while True:
                g = await client.get(f"{base}/task/get", params={"task_id": task_id})
                gj = g.json()
                st = gj.get("state")
                if st in ("completed", "failed"):
                    ROUTER_LOGS.append({"ts": time.time(), "type": "recv", "from": base, "state": st, "task_id": task_id})
                    return gj
                if time.time() - t0 > max_wait:
                    ROUTER_LOGS.append({"ts": time.time(), "type": "timeout", "from": base, "task_id": task_id})
                    return {"state": "failed", "artifacts": [{"parts": [{"type": "text", "text": "(timeout)"}]}]}
                await asyncio.sleep(0.75)

    def pick_trend(text_json: str) -> str:
        # be resilient if model doesn't return valid JSON
        try:
            data = json.loads(text_json)
            trends = data.get("trends") or []
            if trends:
                # pick 1st trend title
                topic = (trends[0].get("topic") or "").strip()
                return topic or "AI in social media"
        except Exception:
            pass
        # fallback: ask the LLM to extract the name
        prompt = (
            "Extract the #1 trend/topic name from the following text. "
            "Respond with ONLY the short name.\n\n"
            f"{text_json[:3000]}"
        )
        return llm_complete(prompt).splitlines()[0].strip()[:80]  # assumes you defined llm_complete

    async def _run_task(task_id: str, user_text: str):
        # 1) Ask trending agent
        t = await store.get(task_id)
        try:
            t.history.append({"ts": time.time(), "event": "host:trending:call"})
            tr = await call_agent(trending_url, "What's trending today?")
            t.history.append({"ts": time.time(), "event": "host:trending:done"})

            artifacts = tr.get("artifacts") or []
            trend_json = ""
            if artifacts and artifacts[-1].get("parts"):
                for p in artifacts[-1]["parts"]:
                    if p.get("type") == "text":
                        trend_json += p.get("text", "")

            # 2) Pick a trend
            topic = pick_trend(trend_json)
            t.history.append({"ts": time.time(), "event": "host:trend:selected", "topic": topic})

            # 3) Ask analyzer
            ar = await call_agent(analyzer_url, f"Analyze the trend: {topic}")
            a_art = ar.get("artifacts") or []
            analysis = ""
            sources = a_art[-1].get("sources", []) if a_art else []
            if a_art and a_art[-1].get("parts"):
                for p in a_art[-1]["parts"]:
                    if p.get("type") == "text":
                        analysis += p.get("text", "")

            out = f"### Selected trend: **{topic}**\n\n{analysis}"
            await store.update(
                task_id,
                state="completed",
                artifacts=[{
                    "parts": [{"type": "text", "text": out}],
                    "sources": sources
                }]
            )
        except Exception as e:
            await store.update(
                task_id,
                state="failed",
                artifacts=[{"parts": [{"type": "text", "text": f"Host error: {e}"}]}]
            )

    @app.post("/message/send")
    async def message_send(req: Request, background_tasks: BackgroundTasks):
        body = await req.json()
        msg = body.get("message") or {}
        parts = msg.get("parts") or []
        text = body.get("query") or (parts[0].get("text") if parts and parts[0].get("type") == "text" else "")
        if not text:
            text = "Analyze current trends and give me a quantified report."

        t = await store.create(context_id=msg.get("contextId", uuid.uuid4().hex))
        background_tasks.add_task(_run_task, t.id, text)
        return {"task_id": t.id, "state": "running"}

    @app.get("/inspector/logs")
    async def inspector_logs():
        return ROUTER_LOGS[-200:]

    return app


In [ ]:

#@title Start all A2A agents (background)
import threading, httpx

nest_asyncio.apply()

WEA_PORT = 10020
ANA_PORT = 10021
HOST_PORT = 10022

trending_app = build_trending_app(WEA_PORT)
analyzer_app = build_analyzer_app(ANA_PORT)
host_app = build_host_app(HOST_PORT, f"http://localhost:{WEA_PORT}", f"http://localhost:{ANA_PORT}")

def run_app(app, port):
    uvicorn.run(app, host="0.0.0.0", port=port, log_level="warning")

threads = [
    threading.Thread(target=run_app, args=(trending_app, WEA_PORT), daemon=True),
    threading.Thread(target=run_app, args=(analyzer_app, ANA_PORT), daemon=True),
    threading.Thread(target=run_app, args=(host_app, HOST_PORT), daemon=True),
]
for th in threads:
    th.start()

time.sleep(2.5)

# quick health checks
async def _check():
    async with httpx.AsyncClient(timeout=5.0) as c:
        a = await c.get(f"http://localhost:{WEA_PORT}/.well-known/agent-card.json")
        b = await c.get(f"http://localhost:{ANA_PORT}/.well-known/agent-card.json")
        h = await c.get(f"http://localhost:{HOST_PORT}/.well-known/agent-card.json")
        return a.json()["name"], b.json()["name"], h.json()["name"]

print("Agents started:", asyncio.run(_check()))
print(f"Trending: http://localhost:{WEA_PORT}")
print(f"Analyzer: http://localhost:{ANA_PORT}")
print(f"Host:     http://localhost:{HOST_PORT}")


Agents started: ('Trending Topics Agent', 'Trend Analyzer Agent', 'Trend Analysis Host')
Trending: http://localhost:10020
Analyzer: http://localhost:10021
Host:     http://localhost:10022


In [ ]:

#@title A2A Inspector UI (Gradio) + Chat
import gradio as gr, httpx, asyncio

AGENTS = {
    "trending": f"http://localhost:{WEA_PORT}",
    "analyzer": f"http://localhost:{ANA_PORT}",
    "host":     f"http://localhost:{HOST_PORT}",
}

async def a2a_send_and_wait(agent_base: str, user_text: str, poll_interval=0.75, max_wait=120.0):
    async with httpx.AsyncClient(timeout=60.0) as client:
        payload = {
            "message": {
                "role": "user",
                "messageId": uuid.uuid4().hex,
                "contextId": uuid.uuid4().hex,
                "parts": [{"type":"text","text": user_text}],
            }
        }
        send_r = await client.post(f"{agent_base}/message/send", json=payload)
        send_j = send_r.json()
        task_id = send_j.get("task_id")
        assert task_id, f"Agent did not return task_id: {send_j}"

        t0 = time.time()
        while True:
            g = await client.get(f"{agent_base}/task/get", params={"task_id": task_id})
            j = g.json()
            state = j.get("state")
            if state == "completed":
                artifacts = j.get("artifacts", [])
                text = ""
                sources = []
                if artifacts:
                    parts = artifacts[-1].get("parts", [])
                    for p in parts:
                        if p.get("type") == "text":
                            text += p.get("text","")
                    sources = artifacts[-1].get("sources", [])
                return {"task_id": task_id, "state": state, "text": text, "sources": sources}
            if state == "failed":
                return {"task_id": task_id, "state": state, "text": "(failed)", "sources": []}
            if time.time() - t0 > max_wait:
                return {"task_id": task_id, "state": "timeout", "text": "(timeout)", "sources": []}
            await asyncio.sleep(poll_interval)

async def chat_fn(message, history):
    ans = await a2a_send_and_wait(AGENTS["host"], message)
    body = ans.get("text","(no content)")
    srcs = ans.get("sources",[])
    if srcs:
        body += "\n\n**Sources**\n" + "\n".join(f"- {u}" for u in srcs)
    return body

async def inspector_data():
    async with httpx.AsyncClient(timeout=15.0) as client:
        cards = {}
        tasks = []
        for name, base in AGENTS.items():
            try:
                cards[name] = (await client.get(f"{base}/.well-known/agent-card.json")).json()
            except:
                cards[name] = {"error":"card fetch failed"}
            try:
                tl = (await client.get(f"{base}/task/list")).json()
                for t in tl:
                    t["agent"] = name
                    tasks.append(t)
            except:
                pass
        try:
            logs = (await client.get(f"{AGENTS['host']}/inspector/logs")).json()
        except:
            logs = []
    return cards, tasks, logs

with gr.Blocks(title="A2A Host + Inspector") as demo:
    gr.Markdown("# A2A Host + Inspector")
    with gr.Tab("Chat with Host"):
        gr.Markdown("We create a **Task** on the Host via `/message/send`, which then orchestrates the Trending and Analyzer agents and returns the final artifact. We poll `/task/get` until `completed`.")
        chat = gr.ChatInterface(fn=chat_fn, type="messages")
    with gr.Tab("Inspector"):
        gr.Markdown("Use the button to view **Agent Cards**, **Tasks** from each service, and **Host logs** (which agent was called, task IDs, timeouts, etc.).")
        btn = gr.Button("Refresh")
        cards_json = gr.JSON(label="Agent Cards")
        tasks_df = gr.Dataframe(headers=["agent","id","context_id","state","created_at","updated_at","history","artifacts"], interactive=False)
        logs_json = gr.JSON(label="Host Orchestrator Logs")
        async def do_refresh():
            c,t,l = await inspector_data()
            return c,t,l
        btn.click(do_refresh, outputs=[cards_json, tasks_df, logs_json])
demo.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ffd2c6e7f324aa4cbd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/langchain_nvidia_ai_endpoints/_common.py:212: UserWarning: Found meta/llama-3.3-70b-instruct in available_models, but type is unknown and inference may fail.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/langchain_nvidia_ai_endpoints/_common.py:212: UserWarning: Found meta/llama-3.3-70b-instruct in available_models, but type is unknown and inference may fail.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/langchain_nvidia_ai_endpoints/_common.py:212: UserWarning: Found meta/llama-3.3-70b-instruct in available_models, but type is unknown and inference may fail.
  warnings.warn(
